In [1]:
import os
from collections import defaultdict
from pathlib import Path
import glob


dataset_path = Path(r"C:\Users\zhou\Desktop\yolo26s")
train_images_dir = dataset_path / "train" / "images"
train_labels_dir = dataset_path / "train" / "labels"
val_images_dir = dataset_path / "valid" / "images"
val_labels_dir = dataset_path / "valid" / "labels"


class_names = {
    0: "crazing",
    1: "inclusion",
    2: "patches",
    3: "pitted_surface",
    4: "rolled-in_scale",
    5: "scratches"
}

# ================== 统计函数 ==================
def count_dataset(split_name, images_dir, labels_dir):
    print(f"\n=== {split_name} 数据集概况 ===")
    
    # 统计图片
    image_files = glob.glob(str(images_dir / "*.jpg")) + \
                  glob.glob(str(images_dir / "*.jpeg")) + \
                  glob.glob(str(images_dir / "*.png"))
    
    # 统计标签
    label_files = glob.glob(str(labels_dir / "*.txt"))
    
    print(f"图片数量: {len(image_files)}")
    print(f"标签文件数量: {len(label_files)}")
    print(f"有标签但无图片的数量: {len(label_files) - len(image_files) if len(label_files) > len(image_files) else 0}")
    
    # 统计每个类别的目标数量
    class_count = defaultdict(int)
    empty_label_count = 0
    total_objects = 0
    
    for label_path in label_files:
        with open(label_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()
            if not lines:
                empty_label_count += 1
                continue
            for line in lines:
                if line.strip():
                    cls_id = int(line.strip().split()[0])
                    class_count[cls_id] += 1
                    total_objects += 1
    
    print(f"总目标框数量: {total_objects}")
    print(f"空标签文件数量（无缺陷）: {empty_label_count}")
    
    print("\n各类别目标数量：")
    for cls_id in range(6):
        count = class_count[cls_id]
        name = class_names[cls_id]
        print(f"  {cls_id:1d} - {name:15s}: {count:6d} 个目标框")
    
    # 简单图片尺寸统计
    sizes = []
    for img_path in image_files[:50]:   
        try:
            from PIL import Image
            with Image.open(img_path) as img:
                sizes.append(img.size)
        except:
            pass
    
    if sizes:
        widths = [w for w, h in sizes]
        heights = [h for w, h in sizes]
        print(f"\n图片尺寸范围: {min(widths)}×{min(heights)} ~ {max(widths)}×{max(heights)}")
        print(f"平均尺寸: {sum(widths)/len(widths):.0f}×{sum(heights)/len(heights):.0f}")

# ================== 执行统计 ==================
print("=== NEU-DET 钢材表面缺陷数据集概况 ===")
print(f"数据集根目录: {dataset_path}\n")

count_dataset("训练集 (train)", train_images_dir, train_labels_dir)
count_dataset("验证集 (valid)", val_images_dir, val_labels_dir)


=== NEU-DET 钢材表面缺陷数据集概况 ===
数据集根目录: C:\Users\zhou\Desktop\yolo26s


=== 训练集 (train) 数据集概况 ===
图片数量: 1770
标签文件数量: 1770
有标签但无图片的数量: 0
总目标框数量: 4125
空标签文件数量（无缺陷）: 0

各类别目标数量：
  0 - crazing        :    681 个目标框
  1 - inclusion      :    996 个目标框
  2 - patches        :    864 个目标框
  3 - pitted_surface :    424 个目标框
  4 - rolled-in_scale:    619 个目标框
  5 - scratches      :    541 个目标框

图片尺寸范围: 200×200 ~ 200×200
平均尺寸: 200×200

=== 验证集 (valid) 数据集概况 ===
图片数量: 30
标签文件数量: 30
有标签但无图片的数量: 0
总目标框数量: 64
空标签文件数量（无缺陷）: 0

各类别目标数量：
  0 - crazing        :      8 个目标框
  1 - inclusion      :     15 个目标框
  2 - patches        :     17 个目标框
  3 - pitted_surface :      8 个目标框
  4 - rolled-in_scale:      9 个目标框
  5 - scratches      :      7 个目标框

图片尺寸范围: 200×200 ~ 200×200
平均尺寸: 200×200

=== 总体总结 ===
类别总数: 6 类
缺陷名称: ['crazing', 'inclusion', 'patches', 'pitted_surface', 'rolled-in_scale', 'scratches']
推荐使用 imgsz=640 进行训练（原图为200x200）


In [2]:
import torch
print("PyTorch 版本:", torch.__version__)
print("CUDA 是否可用:", torch.cuda.is_available())
print("GPU 数量:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("当前 GPU 名称:", torch.cuda.get_device_name(0))
    

PyTorch 版本: 2.5.1
CUDA 是否可用: True
GPU 数量: 1
当前 GPU 名称: NVIDIA GeForce RTX 3080 Ti Laptop GPU


In [3]:
from ultralytics import YOLO

model = YOLO("yolo26s.pt")  

results = model.train(
    data=r"C:\Users\zhou\Desktop\yolo26s\data.yaml", 
    epochs=500,
    imgsz=640,
    batch=16,                    
    device=0,
    patience=50,
    plots=True,
    exist_ok=True,
)


C:\Users\zhou\anaconda3\envs\NEU-DET\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


New https://pypi.org/project/ultralytics/8.4.37 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.36  Python-3.11.15 torch-2.5.1 CUDA:0 (NVIDIA GeForce RTX 3080 Ti Laptop GPU, 16384MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\zhou\Desktop\yolo26s\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=500, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=1.0, multi_sc

In [4]:
# 验证
model.val()


Ultralytics 8.4.36  Python-3.11.15 torch-2.5.1 CUDA:0 (NVIDIA GeForce RTX 3080 Ti Laptop GPU, 16384MiB)
YOLO26s summary (fused): 122 layers, 9,467,502 parameters, 0 gradients, 20.5 GFLOPs
val: Fast image access  (ping: 0.20.2 ms, read: 48.523.1 MB/s, size: 11.4 KB)
val: Scanning C:\Users\zhou\Desktop\yolo26s\valid\labels.cache... 30 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 30/30  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.4s/it 2.7s7.9s
                   all         30         64      0.725      0.757      0.767      0.476
               crazing          5          8      0.672      0.625      0.658      0.317
             inclusion          5         15      0.527      0.667      0.519      0.205
               patches          5         17      0.765      0.882      0.949      0.603
        pitted_surface          5          8      0.884      0.956      0.982      0.695
       rolled-in_scale     

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x00000295D766FC90>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
 